# Notebook 03 — Initial Broad Model

This notebook estimates the initial multiple linear regression model for the cleaned June 2026 DB1C airline ticket sample.

A reproducible 50/50 split is created before model building. The model-building sample is used to estimate the broad baseline model, while the validation sample is reserved for later out-of-sample evaluation.

The notebook focuses on:

- defining the initial modelling variables;
- coding categorical predictors appropriately;
- estimating the broad OLS model;
- assessing overall model fit and coefficient behaviour;
- evaluating potential multicollinearity.

Formal variable reduction, residual diagnostics, model refinement, and validation are deferred to later notebooks.

## 1. Import & Load Prepared Data

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [15]:
PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = (
    PROCESSED_DATA_DIR
    / "db1c_ticket_prepared_202606.parquet"
)

df = pd.read_parquet(DATA_FILE)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

display(df.head())

Rows:    90,086
Columns: 16


,ItinID,TotalAmount,TotalDistance,Coupons,RoundTrip,OnLine,PurchaseWindowGroup,RpCarrier,ItinGeoType,OriginState,Origin,OriginCityMarketID,TaxAmount,AmountExclTax,RpYear,RpMonth
0,20260618618554,398.40,1069.0,1,0,1,21AP,DL,2,FL,FLL,32467,42.12,356.28,2026,6
1,20260610813469,686.81,3180.0,2,1,1,2290,WN,2,IN,IND,32337,76.57,610.24,2026,6
2,20260617427446,644.10,3114.0,3,0,0,2290,DL,2,CA,LGB,32575,78.52,565.58,2026,6
3,2026068607173,306.55,780.0,2,1,1,2290,WN,2,CA,ONT,32575,47.95,258.60,2026,6
4,2026065403481,466.44,3488.0,4,1,1,21AP,F9,2,CA,SFO,32457,70.88,395.56,2026,6


### 1.1 Create Model-Building and Validation Samples

A validation sample is reserved before regression model building begins.

The prepared dataset is randomly divided into:

- **50% model-building sample** — used for the initial broad model, variable reduction, diagnostics, and model refinement;
- **50% validation sample** — kept aside and used only after the final candidate models have been selected.

A fixed random seed is used so that the split is reproducible.

The validation sample will not be used for model-selection, diagnostic, or model-refinement decisions.

In [16]:
SPLIT_SEED = 42
TRAIN_SHARE = 0.50

rng = np.random.default_rng(SPLIT_SEED)

random_order = rng.permutation(len(df))

split_point = int(
    TRAIN_SHARE * len(df)
)

train_idx = random_order[:split_point]
validation_idx = random_order[split_point:]

model_df = (
    df.iloc[train_idx]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    df.iloc[validation_idx]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Full prepared sample:  "
    f"{len(df):,}"
)

print(
    f"Model-building sample: "
    f"{len(model_df):,}"
)

print(
    f"Validation sample:     "
    f"{len(validation_df):,}"
)

print(
    f"Model-building share:  "
    f"{len(model_df) / len(df) * 100:.2f}%"
)

print(
    f"Validation share:      "
    f"{len(validation_df) / len(df) * 100:.2f}%"
)

Full prepared sample:  90,086
Model-building sample: 45,043
Validation sample:     45,043
Model-building share:  50.00%
Validation share:      50.00%


#### Interpretation

The prepared dataset contains 90,086 observations.

A reproducible 50/50 split produces:

- 45,043 observations for model building;
- 45,043 observations for final validation.

The validation sample is reserved from this point onward and will not be used for variable selection, diagnostics, or model refinement.

In [17]:
sample_split_summary = pd.DataFrame({
    "Sample": [
        "Full prepared sample",
        "Model-building sample",
        "Validation sample"
    ],
    "Observations": [
        len(df),
        len(model_df),
        len(validation_df)
    ],
    "Share": [
        1.00,
        len(model_df) / len(df),
        len(validation_df) / len(df)
    ]
})

sample_split_summary.to_csv(
    OUTPUT_DIR / "03_sample_split.csv",
    index=False
)

display(sample_split_summary)

,Sample,Observations,Share
0,Full prepared sample,90086,1.0
1,Model-building sample,45043,0.5
2,Validation sample,45043,0.5


## 2. Define Initial Broad Model

### 2.1 Consolidate Rare Categorical Levels

In [18]:
MIN_CATEGORY_COUNT = 100

# Carrier grouping based only on model-building data
carrier_counts = model_df["RpCarrier"].value_counts()

major_carriers = carrier_counts[
    carrier_counts >= MIN_CATEGORY_COUNT
].index

model_df["RpCarrier_Model"] = model_df["RpCarrier"].where(
    model_df["RpCarrier"].isin(major_carriers),
    "Other"
)

validation_df["RpCarrier_Model"] = validation_df["RpCarrier"].where(
    validation_df["RpCarrier"].isin(major_carriers),
    "Other"
)

# Origin-state grouping based only on model-building data
state_counts = model_df["OriginState"].value_counts()

major_states = state_counts[
    state_counts >= MIN_CATEGORY_COUNT
].index

model_df["OriginState_Model"] = model_df["OriginState"].where(
    model_df["OriginState"].isin(major_states),
    "Other"
)

validation_df["OriginState_Model"] = validation_df["OriginState"].where(
    validation_df["OriginState"].isin(major_states),
    "Other"
)

print("Carrier categories:")
print(model_df["RpCarrier_Model"].value_counts())

print("\nOrigin-state categories:")
print(model_df["OriginState_Model"].value_counts())

print("\nUnseen carrier levels in validation:")
print(
    set(validation_df["RpCarrier_Model"].unique())
    - set(model_df["RpCarrier_Model"].unique())
)

print("\nUnseen state levels in validation:")
print(
    set(validation_df["OriginState_Model"].unique())
    - set(model_df["OriginState_Model"].unique())
)

Carrier categories:
RpCarrier_Model
AA       9009
WN       8948
DL       8408
UA       7408
AS       3541
F9       2622
B6       1928
G4       1725
MX        710
SY        370
Other     204
XP        170
Name: count, dtype: int64

Origin-state categories:
OriginState_Model
CA       5964
FL       4260
TX       4226
NY       2414
IL       2124
CO       1596
VA       1577
GA       1492
WA       1401
NC       1383
PA       1292
AZ       1246
MA       1090
NJ       1015
NV        993
TN        974
MO        890
MI        848
OH        810
MN        804
HI        775
OR        731
Other     612
MD        586
UT        572
KY        566
IN        490
SC        474
WI        397
LA        395
AK        369
PR        348
CT        307
OK        305
ID        226
AL        225
NE        217
MT        216
NM        207
IA        197
AR        186
RI        128
ME        115
Name: count, dtype: int64

Unseen carrier levels in validation:
set()

Unseen state levels in validation:
set()


#### Interpretation

Rare categorical levels are consolidated using frequencies from the model-building sample only, so information from the validation sample does not influence the model specification.

Using the minimum count threshold of 100 observations:

- `RpCarrier_Model` contains 12 modelling categories, including `Other`;
- `OriginState_Model` contains 43 modelling categories, including `Other`.

The `Other` category contains 204 model-building observations for carrier and 612 observations for origin state.

The same grouping rules are then applied to the validation sample. No unseen carrier or origin-state modelling levels remain in the validation data, confirming that the categorical specification can be applied consistently during later model validation.

### 2.2 Reference Categories

Reference categories are selected to provide stable and interpretable comparisons:

- `RoundTrip = 0`
- `OnLine = 1`
- `PurchaseWindowGroup = '21AP'`
- `RpCarrier_Model = 'AA'`
- `ItinGeoType = 2`
- `OriginState_Model = 'CA'`

The coefficients of categorical predictors are therefore interpreted relative to these reference categories, holding the other predictors constant.

### 2.3 Broad Model Specification

The initial broad model includes all candidate predictors identified from the data-preparation and exploratory-analysis stages.

The specification is:

$$
\begin{aligned}
\text{TotalAmount}_i
={}&
\beta_0
+\beta_1\text{TotalDistance}_i
+\beta_2\text{Coupons}_i \\
&+\text{RoundTrip effects}
+\text{OnLine effects}
+\text{PurchaseWindowGroup effects} \\
&+\text{Carrier effects}
+\text{Itinerary geography effects}
+\text{Origin-state effects}
+\varepsilon_i.
\end{aligned}
$$

This model is intentionally broad. No predictor is removed at this stage based on individual coefficient significance.

In [19]:
broad_formula = """
TotalAmount ~
    TotalDistance
    + Coupons
    + C(RoundTrip, Treatment(reference=0))
    + C(OnLine, Treatment(reference=1))
    + C(PurchaseWindowGroup, Treatment(reference='21AP'))
    + C(RpCarrier_Model, Treatment(reference='AA'))
    + C(ItinGeoType, Treatment(reference=2))
    + C(OriginState_Model, Treatment(reference='CA'))
"""

broad_model = smf.ols(
    formula=broad_formula,
    data=model_df
).fit()

print(broad_model.summary())

                            OLS Regression Results                            
Dep. Variable:            TotalAmount   R-squared:                       0.385
Model:                            OLS   Adj. R-squared:                  0.385
Method:                 Least Squares   F-statistic:                     470.2
Date:                Sun, 18 Oct 2026   Prob (F-statistic):               0.00
Time:                        11:08:13   Log-Likelihood:            -3.1710e+05
No. Observations:               45043   AIC:                         6.343e+05
Df Residuals:                   44982   BIC:                         6.348e+05
Df Model:                          60                                         
Covariance Type:            nonrobust                                         
                                                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------

## 3. Initial Broad-Model Results

### 3.1 Interpretation of Categorical Coefficients

Categorical predictors are represented using dummy variables relative to the reference categories defined in Section 2.2.

For example:

- the `RoundTrip = 1` coefficient compares round-trip itineraries with one-way itineraries;
- the `OnLine = 0` coefficient compares `OnLine = 0` with the reference category `OnLine = 1`;
- purchase-window coefficients are interpreted relative to `21AP`;
- carrier coefficients are interpreted relative to `AA`;
- itinerary-geography coefficients are interpreted relative to `ItinGeoType = 2`;
- origin-state coefficients are interpreted relative to `CA`.

Each coefficient represents a conditional association with `TotalAmount`, holding all other predictors in the model constant.

These coefficients should not be interpreted causally because the data are observational.

In [20]:
broad_metrics = pd.DataFrame({
    "Metric": [
        "Observations",
        "Model df",
        "Residual df",
        "R-squared",
        "F-statistic",
        "F-test p-value",
        "SSE",
        "MSE",
        "AIC",
        "BIC"
    ],
    "Value": [
        broad_model.nobs,
        broad_model.df_model,
        broad_model.df_resid,
        broad_model.rsquared,
        broad_model.fvalue,
        broad_model.f_pvalue,
        broad_model.ssr,
        broad_model.mse_resid,
        broad_model.aic,
        broad_model.bic
    ]
})

broad_metrics.to_csv(
    OUTPUT_DIR / "03_broad_model_metrics.csv",
    index=False
)

display(broad_metrics)

,Metric,Value
0,Observations,4.504300e+04
1,Model df,6.000000e+01
2,Residual df,4.498200e+04
3,R-squared,3.854391e-01
4,F-statistic,4.701955e+02
5,F-test p-value,0.000000e+00
6,SSE,3.434705e+09
7,MSE,7.635733e+04
8,AIC,6.343140e+05
9,BIC,6.348456e+05


In [21]:
conf_int = broad_model.conf_int()

broad_coefficients = pd.DataFrame({
    "Variable": broad_model.params.index,
    "Coefficient": broad_model.params.values,
    "Std_Error": broad_model.bse.values,
    "t_stat": broad_model.tvalues.values,
    "p_value": broad_model.pvalues.values,
    "CI_Lower": conf_int[0].values,
    "CI_Upper": conf_int[1].values
})

broad_coefficients.to_csv(
    OUTPUT_DIR / "03_broad_model_coefficients.csv",
    index=False
)

display(broad_coefficients)

,Variable,Coefficient,Std_Error,t_stat,p_value,CI_Lower,CI_Upper
0,Intercept,279.068224,5.723089,48.761817,0.000000e+00,267.850873,290.285574
1,"C(RoundTrip, Treatment(reference=0))[T.1]",140.997328,3.576765,39.420353,0.000000e+00,133.986809,148.007846
2,"C(OnLine, Treatment(reference=1))[T.0]",64.301473,4.822209,13.334443,1.742369e-40,54.849862,73.753084
3,"C(PurchaseWindowGroup, Treatment(reference='21...",-128.765875,2.812424,-45.784655,0.000000e+00,-134.278273,-123.253476
4,"C(PurchaseWindowGroup, Treatment(reference='21...",-137.350623,4.475267,-30.691046,9.803020e-205,-146.122221,-128.579025
...,...,...,...,...,...,...,...
56,"C(OriginState_Model, Treatment(reference='CA')...",7.182823,7.932423,0.905502,3.652045e-01,-8.364859,22.730505
57,"C(OriginState_Model, Treatment(reference='CA')...",-22.979049,8.518000,-2.697705,6.984533e-03,-39.674471,-6.283628
58,"C(OriginState_Model, Treatment(reference='CA')...",31.038812,14.434012,2.150394,3.152938e-02,2.747907,59.329718
59,TotalDistance,0.094401,0.001302,72.512789,0.000000e+00,0.091850,0.096953


### 3.2 Overall Model Fit

The broad model is statistically significant overall:

$$
F = 470.2,
\qquad
p < 0.001.
$$

The model explains approximately 38.5% of the observed variation in airline ticket fares:

$$
R^2 = 0.385.
$$

The model therefore provides meaningful explanatory power for the cross-sectional variation in `TotalAmount`, although a substantial proportion of fare variation remains unexplained by the observable characteristics included in the dataset.

### 3.3 Broad Coefficient Patterns

Several broad patterns are visible in the initial model.

- `TotalDistance` has a positive and highly statistically significant coefficient of approximately 0.094 per additional mile.
- Round-trip itineraries are associated with fares approximately 141 units higher than comparable one-way itineraries, holding the other predictors constant.
- `OnLine = 0` is associated with fares approximately 64 units higher than the reference category `OnLine = 1`.
- Both `2290` and `91UP` purchase-window groups have substantially lower fitted fares than the `21AP` reference category.
- Large differences are observed across reporting carriers, with several carrier coefficients strongly different from the `AA` reference carrier.
- `ItinGeoType = 1` is associated with fares approximately 47 units higher than `ItinGeoType = 2`.
- Origin-state effects are mixed: some states differ significantly from California while many individual state coefficients are not statistically significant.

`Coupons` has a positive coefficient of approximately 3.87, but it is not statistically significant at the 5% level:

$$
t = 1.678,
\qquad
p = 0.093.
$$

Its contribution will therefore be assessed formally during variable reduction rather than being removed at this stage.

Individual dummy-variable p-values are not used to remove categorical predictors. Multi-level categorical predictors will later be assessed jointly as complete predictor blocks.

## 4. Multicollinearity Assessment

Multicollinearity is assessed using the criteria discussed in the lecture:

- pairwise correlation among numerical predictors;
- the relationship between the overall F-test and individual t-tests;
- unexpected coefficient signs;
- variance inflation factors (VIFs).

The purpose is to determine whether strong linear dependence among the predictors is materially affecting coefficient estimation.

### 4.1 Correlation Between Numerical Predictors

In [22]:
numerical_corr = (
    model_df[
        [
            "TotalDistance",
            "Coupons"
        ]
    ]
    .corr()
)

display(numerical_corr)

,TotalDistance,Coupons
TotalDistance,1.000000,0.539062
Coupons,0.539062,1.000000


### 4.2 Variance Inflation Factors

In [23]:
X = pd.DataFrame(
    broad_model.model.exog,
    columns=broad_model.model.exog_names
)

vif_table = pd.DataFrame({
    "Variable": X.columns,
    "VIF": [
        variance_inflation_factor(
            X.values,
            i
        )
        for i in range(X.shape[1])
    ]
})

vif_table = (
    vif_table
    .sort_values(
        "VIF",
        ascending=False
    )
    .reset_index(drop=True)
)

display(vif_table)

,Variable,VIF
0,Intercept,19.321356
1,Coupons,3.006283
2,"C(ItinGeoType, Treatment(reference=2))[T.1]",2.112372
3,TotalDistance,1.943132
4,"C(RpCarrier_Model, Treatment(reference='AA'))[...",1.878549
...,...,...
56,"C(OriginState_Model, Treatment(reference='CA')...",1.039145
57,"C(OriginState_Model, Treatment(reference='CA')...",1.038817
58,"C(OriginState_Model, Treatment(reference='CA')...",1.038572
59,"C(OriginState_Model, Treatment(reference='CA')...",1.036941


In [24]:
vif_no_intercept = vif_table[
    vif_table["Variable"] != "Intercept"
].copy()

print(
    f"Maximum VIF excluding intercept: "
    f"{vif_no_intercept['VIF'].max():.2f}"
)

print(
    f"Mean VIF excluding intercept: "
    f"{vif_no_intercept['VIF'].mean():.2f}"
)

Maximum VIF excluding intercept: 3.01
Mean VIF excluding intercept: 1.31


In [25]:
vif_table.to_csv(
    OUTPUT_DIR / "03_vif_table.csv",
    index=False
)

#### Interpretation

The correlation between the two numerical predictors is:

$$
r(\text{TotalDistance},\text{Coupons}) = 0.539.
$$

This indicates a moderate positive relationship, but not one sufficiently strong on its own to suggest serious multicollinearity.

The VIF results lead to the same conclusion. Excluding the intercept:

- the maximum VIF is approximately 3.01;
- the mean VIF is approximately 1.31;
- `Coupons` has the highest VIF;
- `ItinGeoType` and `TotalDistance` are the next largest, with VIFs of approximately 2.11 and 1.94 respectively.

All VIFs are well below the commonly used threshold of 10 discussed in the lecture.

The overall regression is highly significant, and many individual coefficients are also statistically significant. There is therefore no pattern of a significant overall F-test combined with widespread insignificant t-tests that would indicate severe multicollinearity.

Although the regression output reports a relatively large condition number, the VIF evidence does not indicate a material multicollinearity problem. Differences in the numerical scales of the regressors may also contribute to the condition number.

Accordingly, no predictor is removed from the broad model on multicollinearity grounds.

## 5. Initial Broad-Model Assessment

The initial broad model provides a statistically significant explanation of cross-sectional airline-fare variation.

Key findings are:

- the overall model is highly statistically significant;
- the model explains approximately 38.5% of the variation in `TotalAmount`;
- `TotalDistance`, `RoundTrip`, `OnLine`, `PurchaseWindowGroup`, carrier, itinerary geography, and several origin-state effects show meaningful conditional associations with fare;
- `Coupons` has a positive coefficient but is not statistically significant at the 5% level in the 50% model-building sample;
- multicollinearity does not appear to be a material concern, with a maximum VIF of approximately 3.01 and a mean VIF of approximately 1.31;
- individual insignificant dummy coefficients are not used to remove entire categorical predictors.

No variable is removed in this notebook.

Formal variable reduction is conducted in Notebook 04 using backward selection followed by a detailed comparison of subset models around the model size identified by the automatic procedure.

In [26]:
multicollinearity_summary = pd.DataFrame({
    "Metric": [
        "Correlation: TotalDistance vs Coupons",
        "Maximum VIF excluding intercept",
        "Mean VIF excluding intercept"
    ],
    "Value": [
        numerical_corr.loc["TotalDistance", "Coupons"],
        vif_no_intercept["VIF"].max(),
        vif_no_intercept["VIF"].mean()
    ]
})

multicollinearity_summary.to_csv(
    OUTPUT_DIR / "03_multicollinearity_summary.csv",
    index=False
)

display(multicollinearity_summary)

,Metric,Value
0,Correlation: TotalDistance vs Coupons,0.539062
1,Maximum VIF excluding intercept,3.006283
2,Mean VIF excluding intercept,1.305726
